# 00 — Getting Started with Argo Float Data

This notebook walks you through:
1. Importing the packages we need
2. Downloading data for a single Argo float
3. Understanding what the data looks like
4. Making a simple temperature profile plot

---

## What is an Argo float?

An Argo float is an autonomous ocean instrument that:
- Drifts at a **parking depth** (~1000 m) for ~10 days
- Dives to ~2000 m
- Rises slowly back to the surface, measuring **temperature**, **salinity**, and **pressure**
- Transmits the data via satellite
- Repeats the cycle

Each dive-and-rise is called a **profile** or **cycle**.

Each float is identified by a unique **WMO number**.  
We will use float **6902746** (located in the North Atlantic) throughout these notebooks.
You can find other floats at: https://fleetmonitoring.euro-argo.eu/

## Step 1 — Install packages (run once)

If you followed the setup instructions in the README you can **skip this cell**.  
Otherwise, uncomment and run the line below.

In [ ]:
# Uncomment the line below only if you have NOT run the setup steps in the README
# !pip install argopy pandas matplotlib numpy xarray

## Step 2 — Import packages

We import all the tools we need here so they are available for the rest of the notebook.

In [ ]:
import argopy                              # Argo data downloader
from argopy import DataFetcher as ArgoDataFetcher

import pandas as pd                        # Tables / DataFrames
import numpy as np                         # Numerical operations
import matplotlib.pyplot as plt            # Plotting

print('All packages imported successfully!')
print(f'argopy version: {argopy.__version__}')

## Step 3 — Download data for one float

We use `argopy` to download all available profiles for float **6902746**.  
The data comes from the Argo ERDDAP server — you need an internet connection.

> **Note:** The first download may take 30–60 seconds. Subsequent calls are cached.

In [ ]:
# ---------------------------------------------------------------
# Choose your float — replace this WMO number to use a different float
# ---------------------------------------------------------------
WMO = 3902607

# Download data from the Argo ERDDAP server
print(f'Downloading data for float {WMO} ...')
loader = ArgoDataFetcher(src='gdac').float(WMO)

# argopy returns an xarray Dataset — we convert it to a pandas DataFrame
# which is easier to work with if you are new to Python
ds  = loader.to_xarray()       # xarray Dataset
df  = ds.to_dataframe().reset_index()   # pandas DataFrame

print(f'Download complete!  {len(df):,} data points retrieved.')

## Step 4 — Explore the data

A **DataFrame** is like a spreadsheet.  Each row is one measurement, and each column is one variable.

In [ ]:
# Show the first 5 rows
df.head()

In [ ]:
# Show all column names
print('Columns in the dataset:')
print(df.columns.tolist())

In [ ]:
# Summary statistics for the main oceanographic variables
df[['PRES', 'TEMP', 'PSAL']].describe().round(2)

In [ ]:
# How many profiles (cycles) are in the dataset?
n_cycles = df['CYCLE_NUMBER'].nunique()
print(f'Number of profiles (cycles): {n_cycles}')
print(f'Date range: {df["TIME"].min().date()}  to  {df["TIME"].max().date()}')

### Key variables explained

| Column | Meaning | Unit |
|---|---|---|
| `PRES` | Pressure (≈ depth) | dbar (1 dbar ≈ 1 m) |
| `TEMP` | In-situ temperature | °C |
| `PSAL` | Practical salinity | PSU (dimensionless) |
| `TIME` | Date and time of the profile | UTC |
| `CYCLE_NUMBER` | Profile counter for this float | — |
| `LATITUDE` | Float latitude | degrees North |
| `LONGITUDE` | Float longitude | degrees East |

## Step 5 — Plot a single temperature profile

Let's pick one cycle and plot temperature vs depth.

In [ ]:
# Select the first profile (cycle 1)
cycle_to_plot = 1
profile = df[df['CYCLE_NUMBER'] == cycle_to_plot].sort_values('PRES')

# --- Plot ---
fig, ax = plt.subplots(figsize=(4, 7))

ax.plot(profile['TEMP'], profile['PRES'], color='tomato', linewidth=2)

# Invert y-axis so surface (0 dbar) is at the top
ax.invert_yaxis()

ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Pressure (dbar)', fontsize=12)
ax.set_title(f'Float {WMO} — Cycle {cycle_to_plot}\n{profile["TIME"].iloc[0].date()}',
             fontsize=13)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Step 6 — Plot several profiles together

Plotting multiple cycles on the same axes lets us see seasonal variability.

In [ ]:
# Plot the first 20 cycles, coloured by cycle number
cycles_to_plot = sorted(df['CYCLE_NUMBER'].unique())[:20]

fig, axes = plt.subplots(1, 2, figsize=(10, 7), sharey=True)

# Use a colourmap so each cycle gets a distinct colour
colours = plt.cm.viridis(np.linspace(0, 1, len(cycles_to_plot)))

for colour, cycle in zip(colours, cycles_to_plot):
    profile = df[df['CYCLE_NUMBER'] == cycle].sort_values('PRES')
    axes[0].plot(profile['TEMP'], profile['PRES'], color=colour, alpha=0.8, linewidth=1)
    axes[1].plot(profile['PSAL'], profile['PRES'], color=colour, alpha=0.8, linewidth=1)

for ax in axes:
    ax.invert_yaxis()
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.set_ylabel('Pressure (dbar)', fontsize=11)

axes[0].set_xlabel('Temperature (°C)', fontsize=11)
axes[0].set_title('Temperature profiles', fontsize=12)
axes[1].set_xlabel('Salinity (PSU)', fontsize=11)
axes[1].set_title('Salinity profiles', fontsize=12)

# Add a colourbar to show which cycle is which
sm = plt.cm.ScalarMappable(cmap='viridis',
                            norm=plt.Normalize(vmin=cycles_to_plot[0], vmax=cycles_to_plot[-1]))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, shrink=0.8)
cbar.set_label('Cycle number', fontsize=11)

fig.suptitle(f'Float {WMO} — first {len(cycles_to_plot)} profiles', fontsize=14)
plt.tight_layout()
plt.show()

---

## Summary

You have learned how to:
- Download Argo float data with `argopy`
- Convert the data to a pandas DataFrame
- Inspect the data columns and statistics
- Plot individual and multiple temperature/salinity profiles

**Next notebook:** `01_single_float_contours.ipynb` — visualise how the whole water column
evolves over time using colour-filled contour plots.